# Case Study 4: Sentiment Analysis on Amazon Product Reviews
## Fine-tuned BERT-family Transformer vs LSTM Baseline

**Goal:** classify reviews as negative, neutral, or positive; compare a pretrained transformer fine-tuned with HuggingFace against an LSTM baseline; discuss transfer learning and deployment constraints.

**Label mapping:** 1–2 stars = negative (0), 3 stars = neutral (1), 4–5 stars = positive (2). We use DistilBERT, a compact BERT-family pretrained transformer, to make the classroom experiment more manageable.


In [ ]:
# If needed:
# %pip install -r ../requirements.txt
import random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## 1. Load Amazon reviews from HuggingFace Datasets
Sample sizes are configurable. Larger samples can improve results but require more training time and memory. Internet access is needed to download the dataset and pretrained model.


In [ ]:
TRAIN_LIMIT, VALID_LIMIT, TEST_LIMIT = 3000, 600, 600
MAX_LENGTH = 128
MODEL_NAME = "distilbert-base-uncased"
LABEL_NAMES = ["negative", "neutral", "positive"]

dataset = load_dataset("amazon_reviews_multi", "en")
def label_stars(row):
    stars = int(row["stars"])
    label = 0 if stars <= 2 else (1 if stars == 3 else 2)
    return {"text": row["review_body"], "label": label}

train_ds = dataset["train"].map(label_stars, remove_columns=dataset["train"].column_names)
valid_ds = dataset["validation"].map(label_stars, remove_columns=dataset["validation"].column_names)
test_ds = dataset["test"].map(label_stars, remove_columns=dataset["test"].column_names)

def clean_sample(ds, limit):
    ds = ds.filter(lambda row: row["text"] is not None and len(row["text"].strip()) > 0)
    ds = ds.shuffle(seed=SEED)
    return ds.select(range(min(limit, len(ds))))

train_ds = clean_sample(train_ds, TRAIN_LIMIT)
valid_ds = clean_sample(valid_ds, VALID_LIMIT)
test_ds = clean_sample(test_ds, TEST_LIMIT)
print("Train/validation/test:", len(train_ds), len(valid_ds), len(test_ds))
display(pd.DataFrame({"text":train_ds["text"][:5],"label":[LABEL_NAMES[i] for i in train_ds["label"][:5]]}))


## 2. Evaluation metrics
Macro-averaged precision, recall, and F1 assign equal weight to each class. This is useful because a single accuracy score can hide weak performance on one sentiment class.


In [ ]:
def metrics(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return {"accuracy":accuracy_score(y_true,y_pred), "macro_precision":p, "macro_recall":r, "macro_f1":f}

def show_results(y_true, y_pred, name):
    print(name)
    print(classification_report(y_true,y_pred,target_names=LABEL_NAMES,zero_division=0))
    print(metrics(y_true,y_pred))
    cm = confusion_matrix(y_true,y_pred,labels=[0,1,2])
    ConfusionMatrixDisplay(cm,display_labels=LABEL_NAMES).plot(cmap="Blues",values_format="d")
    plt.title(name+" confusion matrix"); plt.tight_layout(); plt.show()


## 3. Fine-tune pretrained DistilBERT using HuggingFace Trainer
The pretrained transformer already has general language representations. Fine-tuning adapts them to sentiment classification using our labeled reviews.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

tr = train_ds.map(tokenize, batched=True).remove_columns(["text"])
va = valid_ds.map(tokenize, batched=True).remove_columns(["text"])
te = test_ds.map(tokenize, batched=True).remove_columns(["text"])
collator = DataCollatorWithPadding(tokenizer=tokenizer)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3,
    id2label={i:n for i,n in enumerate(LABEL_NAMES)},
    label2id={n:i for i,n in enumerate(LABEL_NAMES)}
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return metrics(labels, np.argmax(logits,axis=-1))

args = TrainingArguments(
    output_dir="./bert_results",
    evaluation_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    seed=SEED,
    fp16=torch.cuda.is_available()
)
trainer = Trainer(model=model,args=args,train_dataset=tr,eval_dataset=va,
                  data_collator=collator,compute_metrics=compute_metrics,tokenizer=tokenizer)

start = time.time()
trainer.train()
bert_seconds = time.time()-start
pred_out = trainer.predict(te)
bert_pred = np.argmax(pred_out.predictions,axis=-1)
y_test = np.array(te["label"])
bert_metrics = metrics(y_test,bert_pred)
print("Fine-tuning seconds:",round(bert_seconds,1))
show_results(y_test,bert_pred,"Fine-tuned DistilBERT")


## 4. Train an LSTM baseline from scratch
The baseline uses the same tokenizer's token IDs but learns its own embedding and bidirectional LSTM. It is a compact baseline rather than a pretrained language model.


In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, ds):
        self.texts, self.labels = list(ds["text"]), list(ds["label"])
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        enc = tokenizer(self.texts[i],truncation=True,max_length=MAX_LENGTH,
                        padding="max_length",return_tensors="pt")
        return {"input_ids":enc["input_ids"].squeeze(0),
                "mask":enc["attention_mask"].squeeze(0),
                "label":torch.tensor(int(self.labels[i]),dtype=torch.long)}

class LSTMClassifier(nn.Module):
    def __init__(self,vocab_size,pad_id,embed_dim=128,hidden=96):
        super().__init__()
        self.emb=nn.Embedding(vocab_size,embed_dim,padding_idx=pad_id)
        self.lstm=nn.LSTM(embed_dim,hidden,batch_first=True,bidirectional=True)
        self.drop=nn.Dropout(0.3)
        self.fc=nn.Linear(hidden*2,3)
    def forward(self,input_ids,mask):
        out,_=self.lstm(self.emb(input_ids))
        m=mask.unsqueeze(-1).float()
        pooled=(out*m).sum(1)/m.sum(1).clamp(min=1)
        return self.fc(self.drop(pooled))

train_loader=DataLoader(ReviewDataset(train_ds),batch_size=32,shuffle=True)
valid_loader=DataLoader(ReviewDataset(valid_ds),batch_size=32)
test_loader=DataLoader(ReviewDataset(test_ds),batch_size=32)
lstm=LSTMClassifier(len(tokenizer),tokenizer.pad_token_id).to(DEVICE)
optimizer=torch.optim.Adam(lstm.parameters(),lr=1e-3)
loss_fn=nn.CrossEntropyLoss()

def run_epoch(loader,training=False):
    lstm.train(training)
    total_loss=0; ys=[]; ps=[]
    for batch in loader:
        ids=batch["input_ids"].to(DEVICE); mask=batch["mask"].to(DEVICE); y=batch["label"].to(DEVICE)
        if training: optimizer.zero_grad()
        with torch.set_grad_enabled(training):
            logits=lstm(ids,mask); loss=loss_fn(logits,y)
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(lstm.parameters(),1.0)
                optimizer.step()
        total_loss += loss.item()*len(y)
        ys.extend(y.detach().cpu().tolist()); ps.extend(logits.argmax(1).detach().cpu().tolist())
    return total_loss/max(len(ys),1), np.array(ys), np.array(ps)

start=time.time()
for epoch in range(3):
    train_loss,_,_=run_epoch(train_loader,True)
    val_loss,val_y,val_p=run_epoch(valid_loader,False)
    print(f"Epoch {epoch+1}/3 | train loss {train_loss:.4f} | validation loss {val_loss:.4f} | validation accuracy {accuracy_score(val_y,val_p):.4f}")
lstm_seconds=time.time()-start
_,lstm_y,lstm_pred=run_epoch(test_loader,False)
lstm_metrics=metrics(lstm_y,lstm_pred)
print("LSTM training seconds:",round(lstm_seconds,1))
show_results(lstm_y,lstm_pred,"LSTM baseline")


## 5. Compare results
These results are generated by your own run. Do not invent or copy scores from another computer; runtime and metrics vary with sample size, hardware, and package versions.


In [ ]:
comparison=pd.DataFrame([
    {"Model":"Fine-tuned DistilBERT",**bert_metrics,"training_seconds":bert_seconds},
    {"Model":"LSTM baseline",**lstm_metrics,"training_seconds":lstm_seconds}
])
display(comparison.round(4))
comparison.set_index("Model")[["accuracy","macro_precision","macro_recall","macro_f1"]].plot(kind="bar",figsize=(10,5),ylim=(0,1))
plt.title("DistilBERT vs LSTM"); plt.ylabel("Score"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()


## 6. Test a custom review
The prediction is model output, not guaranteed truth.


In [ ]:
def predict_sentiment(text):
    model.eval()
    batch=tokenizer(text,return_tensors="pt",truncation=True,max_length=MAX_LENGTH)
    batch={k:v.to(model.device) for k,v in batch.items()}
    with torch.no_grad():
        probs=torch.softmax(model(**batch).logits,dim=-1)[0].cpu().numpy()
    return {"sentiment":LABEL_NAMES[int(np.argmax(probs))],
            "probabilities":{LABEL_NAMES[i]:round(float(probs[i]),4) for i in range(3)}}

print(predict_sentiment("The product is excellent and works exactly as expected."))
print(predict_sentiment("The quality is poor and it stopped working after one day."))


## 7. Discussion: transfer learning and deployment constraints

**Transfer learning efficiency**
- The pretrained transformer starts with language representations learned from large text corpora.
- Fine-tuning adapts those representations to the review labels and can need less task-specific data than training a transformer from scratch.
- DistilBERT is a smaller BERT-family model intended to reduce cost relative to larger transformers.
- The LSTM here is trained from scratch, so the comparison is illustrative; architecture and pretraining both affect results.

**Deployment constraints**
- Latency and throughput: measure inference time on target hardware.
- Memory and compute: model size affects RAM/VRAM and hosting cost.
- Privacy: avoid unnecessary storage of personal information in review text.
- Class imbalance: inspect per-class recall/F1 and confusion matrix, not accuracy alone.
- Drift: product language and customer preferences change over time.
- Ambiguity: sarcasm, mixed opinions, and context can cause errors.
- Monitoring: define quality thresholds, logging, and periodic reevaluation before production.

**Conclusion questions:** Which model has higher macro-F1? Which class is hardest to predict? Which trained faster on your hardware? What are the trade-offs between performance and deployment cost?
